# Benchmark SLM (prompts maison + HumanEval)

Notebook **commun aux 4 membres**. Vous ne changez que `MODEL_KEY` (cellule 3) et, une seule fois, `REPO_URL` / `REPO_NAME` (cellule 1).

Avant de lancer : `Exécution > Modifier le type d'exécution > T4 GPU`.

- **Partie A** : 15 prompts maison -> `results/benchmark_<modele>.csv`
- **Partie B** : 20 problèmes HumanEval -> `results/humaneval_<modele>.json`

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU only, same as the other members
REPO_URL = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH = "correctifs-gemma"   # branch that holds the corrected benchmark files

if not os.path.exists("../config.json"):
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

os.makedirs("../results", exist_ok=True)   # Git does not keep empty folders


Cloning into 'SLM-Software-Engineering'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 44 (delta 11), reused 18 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 18.27 KiB | 263.00 KiB/s, done.
Resolving deltas: 100% (11/11), done.
/kaggle/working/SLM-Software-Engineering/benchmarking/notebooks


In [3]:
!pip install -r ../requirements.txt -q
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-8021724d-ea85-b099-9177-75bfc024468a)
GPU 1: Tesla T4 (UUID: GPU-969412d3-e986-3965-5537-4e3f762c9b56)


## Gemma only
Gated model: only Gemma needs a Hugging Face token. **Qwen2.5-Coder is open (Apache 2.0): nothing to do here.**


In [ ]:
# Qwen does not need a Hugging Face login. (Gemma only: login with the Kaggle secret HF_TOKEN.)


In [4]:
import json, time, sys, re
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

# === SEULE LIGNE A CHANGER : "qwen", "phi", "gemma" ou "smollm" ===
MODEL_KEY = "qwen"
# ====================================================================

with open("../config.json") as f:
    config = json.load(f)

MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)   # /no_think pour SmolLM3 seulement
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
print(MODEL_NAME, "|", model_slug, "|", GPU_NAME)

Qwen/Qwen2.5-Coder-1.5B-Instruct | qwen2.5-coder-1.5b-instruct | Tesla T4


In [5]:
torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
DTYPE = torch.bfloat16 if config["precision"] == "bf16" else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=DTYPE
)
print("Précision :", config["precision"])
model.eval()


def generate(user_prompt: str):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})

    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]

    # TTFT : temps pour produire 1 seul token
    torch.cuda.synchronize(); t0 = time.time()
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.time() - t0

    # Generation complete
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                         do_sample=GEN_PARAMS["do_sample"])
    torch.cuda.synchronize(); latency = time.time() - t0

    new_tokens = out[0][n_in:]
    return {
        "response": tokenizer.decode(new_tokens, skip_special_tokens=True),
        "gpu": GPU_NAME,
        "ttft_s": round(ttft, 3),
        "latency_s": round(latency, 3),
        "tokens_per_sec": round(len(new_tokens) / latency, 2) if latency > 0 else None,
        "input_tokens": n_in,
        "output_tokens": len(new_tokens),
        "gpu_memory_go": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


# Échauffement + contrôle : le 1er appel est lent, et on vérifie que le modèle répond vraiment
test = generate("Write a Python function add(a, b) that returns a + b.")
print(test["response"][:300])
assert "def" in test["response"], "Réponse vide ou incohérente : essayez bf16 dans config.json, puis redémarrez la session."
print("Modele pret.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Précision : bf16
```python
def add(a, b):
    return a + b
```

In this solution, the function `add` takes two parameters, `a` and `b`, and returns their sum. The function is straightforward and utilizes Python's built-in addition operator (`+`) to perform the operation. This function can be used in any Python progr
Modele pret.


## Partie A : 15 prompts maison

In [6]:
with open("../prompts.json") as f:
    prompts = json.load(f)["prompts"]

def norm(s):  # pour comparer sans tenir compte des espaces
    return re.sub(r"\s+", "", s)

results = []
for p in prompts:
    gen = generate(p["prompt"])
    response = gen.pop("response")

    status, error = None, None
    if p["task"] == "generation_code" and "test" in p:
        status, error = run_tests(extract_code(response), p["test"],
                                  timeout=HE_CONFIG["timeout_seconds"])
    elif p["task"] == "completion_code" and "expected" in p:
        status = "pass" if norm(p["expected"]) in norm(response) else "fail"
    # bug_detection, explication, refactoring, generation_tests : notes manuelles

    results.append({
        "id": p["id"], "task": p["task"], "prompt": p["prompt"], "response": response,
        **gen,
        "auto_status": status, "auto_error": error,
        # A remplir a la main (voir grille_notation.md et README)
        "bug_label": None, "note_qualite": None, "hallucination": None,
    })
    print(p["id"], status)

df_prompts = pd.DataFrame(results)
df_prompts.to_csv(f"../results/benchmark_{model_slug}.csv", index=False)
print(f"-> results/benchmark_{model_slug}.csv")
df_prompts[["id", "task", "ttft_s", "latency_s", "tokens_per_sec", "gpu_memory_go", "auto_status"]]

P1 pass
55
P2 pass
monde le Bonjour
P3 pass
3
P4 pass
P5 fail
P6 pass
P7 None
P8 None
P9 None
P10 None
P11 None
P12 None
P13 None
P14 None
P15 None
-> results/benchmark_qwen2.5-coder-1.5b-instruct.csv


,id,task,ttft_s,latency_s,tokens_per_sec,gpu_memory_go,auto_status
0,P1,generation_code,0.243,10.664,25.41,3.11,pass
1,P2,generation_code,0.109,15.171,25.77,3.12,pass
2,P3,generation_code,0.108,12.129,26.30,3.12,pass
3,P4,generation_code,0.109,13.971,26.05,3.12,pass
4,P5,completion_code,0.111,5.852,26.14,3.11,fail
5,P6,completion_code,0.148,5.643,25.34,3.11,pass
6,P7,bug_detection,0.111,5.859,25.60,3.11,None
7,P8,bug_detection,0.152,5.722,25.86,3.11,None
8,P9,bug_detection,0.112,5.398,25.93,3.11,None
9,P10,explication,0.154,3.217,25.49,3.11,None


## Partie B : HumanEval (20 problèmes communs)

In [7]:
with open("../humaneval_subset.json") as f:
    subset = json.load(f)

humaneval = load_dataset(HE_CONFIG["dataset"], split="test")
humaneval = humaneval.filter(lambda ex: ex["task_id"] in subset["task_ids"])
print(len(humaneval), "problemes")

HEADER = "from typing import *\nimport math\n\n"   # imports courants que le modele peut oublier

details = []
for ex in humaneval:
    gen = generate(HE_CONFIG["instruction"] + ex["prompt"])
    code_generated = extract_code(gen["response"])
    test_code = ex["test"] + f"\ncheck({ex['entry_point']})"
    status, error = run_tests(HEADER + ex["prompt"] + "\n" + code_generated, test_code,
                              timeout=HE_CONFIG["timeout_seconds"])
    details.append({
        "task_id": ex["task_id"], "status": status, "error": error,
        "ttft_s": gen["ttft_s"], "latency_s": gen["latency_s"],
        "tokens_per_sec": gen["tokens_per_sec"],
        "output_tokens": gen["output_tokens"], "gpu_memory_go": gen["gpu_memory_go"],
        "response": gen["response"],
    })
    print(ex["task_id"], status)

n_pass = sum(d["status"] == "pass" for d in details)
summary = {
    "model": MODEL_NAME,
    "gpu": GPU_NAME,
    "precision": config["precision"],
    "dataset": "HumanEval (sous-ensemble des 20 premiers problemes)",
    "n_problems": len(details),
    "n_pass": n_pass,
    "pass@1": round(100 * n_pass / len(details), 1),
    "details": details,
}
with open(f"../results/humaneval_{model_slug}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"pass@1 = {summary['pass@1']}% ({n_pass}/{len(details)}) -> results/humaneval_{model_slug}.json")

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/164 [00:00<?, ? examples/s]

20 problemes
HumanEval/0 pass
HumanEval/1 fail
HumanEval/2 pass
HumanEval/3 pass
HumanEval/4 pass
HumanEval/5 pass
HumanEval/6 fail
HumanEval/7 pass
HumanEval/8 pass
HumanEval/9 pass
HumanEval/10 fail
HumanEval/11 pass
HumanEval/12 pass
HumanEval/13 pass
HumanEval/14 pass
HumanEval/15 pass
HumanEval/16 pass
HumanEval/17 pass
HumanEval/18 pass
HumanEval/19 fail
pass@1 = 80.0% (16/20) -> results/humaneval_qwen2.5-coder-1.5b-instruct.json


In [11]:
# Fichier d'environnement PROPRE A CHAQUE MEMBRE (pas d'ecrasement entre vous)
from importlib.metadata import version
import platform
lines = [f"# Environnement - {MODEL_NAME}", "",
         f"- GPU : {GPU_NAME}",
         f"- Python : {platform.python_version()}",
         f"- torch : {torch.__version__}",
         f"- CUDA : {torch.version.cuda}"]
for lib in ["transformers", "accelerate", "huggingface_hub", "datasets", "pandas"]:
    lines.append(f"- {lib} : {version(lib)}")
open(f"../environnement_{model_slug}.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

# Environnement - Qwen/Qwen2.5-Coder-1.5B-Instruct

- GPU : Tesla T4
- Python : 3.13.15
- torch : 2.11.0+cu128
- CUDA : 12.8
- transformers : 5.16.1
- accelerate : 1.14.0
- huggingface_hub : 1.29.0
- datasets : 4.8.5
- pandas : 2.3.3
